# Task 2: Train the Dynamic Sign Classifier

Loads all clip CSVs collected in `02_data_collection.ipynb` (from
`../data/raw/`), trains a classifier on the resampled gesture sequences, and
evaluates it (accuracy, per-class precision/recall, confusion matrix).

Run cells top to bottom. Kernel: **Python (ipcv-task2)**.

## 1. Load and merge all collected CSVs

In [ ]:
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

csv_files = sorted(glob.glob("../data/raw/*.csv"))
print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(" ", f)

if not csv_files:
    raise FileNotFoundError(
        "No CSVs found in ../data/raw/. Run 02_data_collection.ipynb first."
    )

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Vocabulary actually used (15 words). Chosen with scripts/screen_stage1.py, evaluate_words.py
# --split source and select_words.py: every word was tested on recording sources the model
# never trained on, and added only if it passed the bar. See README, 'Final vocabulary'. Set to None to train on all
# 25 dataset words again. "Idle" is always kept.
KEEP_WORDS = ["Apple", "Please", "Color", "Yes", "No", "More", "Boy", "Hello", "Bathroom", "Yesterday", "Brown", "Football", "Water", "Red"]
if KEEP_WORDS:
    df = df[df["label"].isin(KEEP_WORDS + ["Idle"])].reset_index(drop=True)
print(f"\nTotal samples: {len(df)}")
print(df["label"].value_counts())

## 2. Prepare features and labels

In [ ]:
feature_cols = [c for c in df.columns if c.startswith("feature_")]
print(f"Feature vector length: {len(feature_cols)}  (should be TARGET_FRAMES * 63)")

X = df[feature_cols].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

## 3. Train a baseline classifier

Random Forest on the flattened, resampled gesture sequences. No need for an
LSTM/sequence model here — fixed-length resampling already turns each clip
into a fixed-size vector, so a standard classifier works directly on it.

In [ ]:
# class_weight: Idle (recorded by hand) has far fewer clips than each dataset sign.
# Without weighting, 14 Idle clips were recognised only 14% of the time; with it, 36%
# (and still only 2 of 1,348 sign clips were wrongly called Idle).
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1,
                             class_weight="balanced_subsample")
clf.fit(X_train, y_train)

train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy:  {test_acc:.3f}")

## 4. Evaluate: per-class report + confusion matrix

This is the evidence your Task 2 report section needs: which signs get
confused with which, and per-class precision/recall.

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
fig, ax = plt.subplots(figsize=(13, 13))
disp.plot(ax=ax, xticks_rotation=60, colorbar=False)
plt.title("Confusion Matrix — Dynamic Sign Classifier")
plt.tight_layout()
plt.savefig("../data/confusion_matrix.png", dpi=150)
plt.show()
print("Saved plot to ../data/confusion_matrix.png (useful for your report/poster)")
print("NOTE: this split is random, so near-duplicate clips from the same signer/session can land")
print("on both sides — treat it as an optimistic number. The domain-gap check below is stricter.")

## 4b. Domain-gap check (the honest number for your demo)

Trains **only on dataset videos** (`collector == "hf_dataset"`) and tests on
**your own webcam recordings** (everything else in `../data/raw/`). This is
the closest thing to "how will it do on demo day with a real person in front
of my webcam?" — report this number, not just the random-split accuracy above.

Needs at least one webcam CSV from `02_data_collection.ipynb` recorded with
the *current* feature pipeline.

In [ ]:
is_ext = (df["collector"] == "hf_dataset").values

if is_ext.any() and (~is_ext).any():
    gap_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    gap_clf.fit(X[is_ext], y[is_ext])

    X_own, y_own = X[~is_ext], y[~is_ext]
    known = np.isin(y_own, gap_clf.classes_)  # e.g. "Idle" isn't in the dataset
    if known.sum() == 0:
        print(f"Skipping: your {len(y_own)} webcam clips are all labels the dataset lacks "
              f"({sorted(set(y_own))}), so there is nothing to test the dataset-trained model on.")
        print("This check only applies if you record some of your own SIGN clips "
              "(set SIGNS = ALL_SIGNS + [\"Idle\"] in 02_data_collection.ipynb). It is optional.")
    else:
        pred_own = gap_clf.predict(X_own[known])
        print(f"Webcam clips tested: {known.sum()} (skipped {(~known).sum()} with labels the dataset lacks, e.g. Idle)")
        print(f"Dataset-trained model on YOUR webcam clips: {accuracy_score(y_own[known], pred_own):.3f}")
        print(classification_report(y_own[known], pred_own, zero_division=0))
else:
    print("Need both dataset clips and your own webcam clips in ../data/raw/ to run this check.")
    print("Record some with 02_data_collection.ipynb first (even ~10 clips per sign is enough).")

## 5. Feature importance (optional, useful for Q&A)

Shows which parts of the resampled sequence the model relies on most — useful
if a juror asks "how does your classifier actually distinguish these signs?"

In [ ]:
importances = clf.feature_importances_
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

plt.figure(figsize=(8, 5))
plt.barh(range(top_n), importances[top_idx][::-1])
plt.yticks(range(top_n), [feature_cols[i] for i in top_idx][::-1], fontsize=7)
plt.xlabel("Feature importance")
plt.title(f"Top {top_n} most important features")
plt.tight_layout()
plt.show()

## 6. Save the trained model

Saves the classifier plus the config needed to reproduce the exact same
feature pipeline at inference time (target sequence length, class list).

In [ ]:
TARGET_FRAMES = len(feature_cols) // 63
assert len(feature_cols) % 63 == 0, "feature_cols length should be a multiple of 63"

model_bundle = {
    "model": clf,
    "classes": list(clf.classes_),
    "target_frames": TARGET_FRAMES,
    "feature_dim_per_frame": 63,
}

out_path = "../models/sign_classifier.joblib"
joblib.dump(model_bundle, out_path)
print(f"Saved model bundle to {out_path}")
print(f"Classes: {model_bundle['classes']}")
print(f"Target frames per clip: {TARGET_FRAMES}")

## Next steps

1. If accuracy is low or specific signs are heavily confused (check the
   confusion matrix), collect more clips for those signs in
   `02_data_collection.ipynb`, then re-run this notebook.
2. Move to `04_live_recognition.ipynb` to test the trained model live with a
   rolling buffer, motion gating, and confirmation/cooldown logic — this is
   the actual real-time behavior Task 2 needs to hand off to Task 3/4.